# MWIZERWA AI — Google Colab

Run cells **in order** with **Runtime → GPU**.

| Step | Purpose |
|------|---------|
| 1 | Clone/use `https://github.com/Mindora-AI/ai_engine` at `/content/ai_engine` |
| 2 | Mount Drive → `MyDrive/MWIZERWA` |
| 3 | Hardware inspection |
| 4 | Python dependencies |
| 5 | CUDA-aware `llama-cpp-python` + GPU layers |
| 6 | DeepKIN (KinyaFlex) |
| 7 | Model check & download |
| 8 | Real validation |
| 9 | Gradio UI (only if validation passes) |

Private repo: set env `MWIZERWA_REPO_URL` before step 1 (use Colab Secrets, not hardcoded tokens).

Docs: `docs/COLAB.md`

## 1. Repository initialization

Success: prints path to existing checkout or clones into `/content/ai_engine`.

Failure: non-empty folder without `run_assistant.py` — rename/remove manually.

In [ ]:
import os
import sys
from pathlib import Path

PROJECT = Path(os.environ.get('MWIZERWA_COLAB_PROJECT', '/content/ai_engine'))
MARKER = PROJECT / 'run_assistant.py'

if not MARKER.is_file():
    if PROJECT.is_file():
        raise RuntimeError(f'{PROJECT} is a file, not a directory')
    if PROJECT.exists() and any(PROJECT.iterdir()):
        raise RuntimeError(
            f'{PROJECT} exists but is not a MWIZERWA checkout. '
            'Remove/rename it or set MWIZERWA_COLAB_PROJECT.'
        )
    import subprocess
    url = os.environ.get('MWIZERWA_REPO_URL', 'https://github.com/Mindora-AI/ai_engine.git')
    PROJECT.parent.mkdir(parents=True, exist_ok=True)
    print('Cloning', url)
    subprocess.run(['git', 'clone', '--depth', '1', url, str(PROJECT)], check=True)
else:
    print('Using existing project:', PROJECT)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
os.environ['MWIZERWA_PROJECT_ROOT'] = str(PROJECT)

from scripts.colab_runtime import require_colab_scripts
require_colab_scripts(PROJECT)
print('MWIZERWA_PROJECT_ROOT =', PROJECT)

## 2. Google Drive persistence

Success: env vars `MWIZERWA_MODELS_DIR`, `MWIZERWA_CACHE_DIR`, `MWIZERWA_OUTPUT_DIR`, `HF_HOME` printed.

If `drive.mount` fails, fix authentication here — later cells will not recover.

Creating folders does **not** mean model files exist yet (step 7).

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/MWIZERWA')

from scripts.colab_runtime import configure_drive_persistence, report_model_asset_status

configure_drive_persistence(DRIVE_ROOT, PROJECT)
report_model_asset_status()

## 3. Hardware inspection

Inspect RAM, disk, and GPU before installing heavy packages.

In [ ]:
import json
from scripts.colab_runtime import inspect_hardware

HW = inspect_hardware()
print(json.dumps(HW, indent=2))

## 4. Dependencies

Installs `requirements-colab.txt`. **llama-cpp** is installed in the next cell (CUDA-aware).

In [ ]:
from scripts.colab_runtime import install_python_dependencies

install_python_dependencies(PROJECT)

## 5. llama-cpp-python (CUDA when available)

Tries a wheel matching `torch.version.cuda`, then fallbacks `cu124`…`cu121`.

- Success (CUDA): message `CUDA offload available`.
- CPU fallback: explicit warning; `MWIZERWA_LLM_N_GPU_LAYERS=0`.

In [ ]:
from scripts.colab_runtime import (
    install_llama_cpp_for_colab,
    apply_llm_gpu_layers,
    llama_supports_gpu_offload,
)

LLAMA_INFO = install_llama_cpp_for_colab()
print(LLAMA_INFO)
print('llama_supports_gpu_offload():', llama_supports_gpu_offload())
apply_llm_gpu_layers(HW, LLAMA_INFO)

## 6. DeepKIN (KinyaFlex)

Required for Kinyarwanda TTS only.

In [ ]:
from scripts.colab_runtime import install_deepkin, verify_critical_imports

install_deepkin(PROJECT)
verify_critical_imports()

## 7. Models — verify then download

`--check` must pass before download (no `|| true`).

Reuses valid files on Drive; downloads only missing/invalid assets.

In [ ]:
import subprocess
import sys

def run_py(args):
    print('>>', ' '.join(args))
    r = subprocess.run(args, cwd=PROJECT)
    if r.returncode != 0:
        raise RuntimeError(f'Command failed (exit {r.returncode}): {" ".join(args)}')

py = sys.executable
run_py([py, 'scripts/download_models.py', '--check'])
run_py([py, 'scripts/download_models.py'])
run_py([py, 'scripts/download_models.py', '--check'])

from scripts.colab_runtime import report_model_asset_status
report_model_asset_status()

## 8. Real validation

LLM (rw/en/fr), KinyaFlex + Piper TTS, optional ASR.

Must exit 0 before Gradio.

In [ ]:
import subprocess
import sys

r = subprocess.run([sys.executable, 'scripts/colab_validate.py'], cwd=PROJECT)
if r.returncode != 0:
    raise RuntimeError(
        'Validation failed. Fix models/deps above — Gradio will not start.'
    )
MWIZERWA_VALIDATION_OK = True
print('Validation OK — safe to launch Gradio.')

## 9. Gradio UI

Uses `ConversationEngine`, safety gates, and TTS router (KinyaFlex/Piper).

Audio on Drive: `MyDrive/MWIZERWA/outputs/tts_responses/latest_response.wav`

In [ ]:
if not globals().get('MWIZERWA_VALIDATION_OK'):
    raise RuntimeError('Run the validation cell successfully first.')

!python scripts/colab_gradio.py